# SVCM — ITI-98 — Détails CodeSystem ($lookup)

**Profil** : IHE ITI SVCM
**Acteur testé** : SVCM-Terminology Consumer
**Transaction** : SVCM-ITI-98
**Référence Gazelle** : https://interop.esante.gouv.fr/tm/testing/testsDefinition/viewTestPage.seam?id=12209&cid=48786

Trois scénarios :
1. GET `$lookup` du code CIM-11 `DA63` — vérifier libellé "Ulcère du duodénum"
2. GET `$lookup` avec propriété `child` — afficher les codes enfants
3. POST `$lookup` du code SNOMED `425803006` — vérifier si inactif

## Configuration

In [1]:
import requests
import json
import time
import os, getpass
from datetime import datetime
from urllib.parse import quote

FHIR_BASE = "https://gazelle-proxypatans.kereval.cloud:10102/fhir"
#FHIR_BASE = os.environ.get("FHIR_BASE", "https://smt.esante.gouv.fr/fhir")

HTTP_RETRIES = 3
HTTP_BACKOFF = 2

HEADERS_JSON = {
    "Accept": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
HEADERS_POST = {
    "Accept": "application/fhir+json",
    "Content-Type": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
HEADERS_API = {
    "accept": "*/*",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}

# ── run output directory ──────────────────────────────────────────────────────
NOTEBOOK_ID = "05_iti98_lookup"
RUN_TS  = datetime.now().strftime("%Y%m%dT%H%M%S")
RUN_DIR = os.path.join("runs", f"{NOTEBOOK_ID}_{RUN_TS}")
os.makedirs(RUN_DIR, exist_ok=True)

# ── helpers ───────────────────────────────────────────────────────────────────

def http_get(url, headers=None):
    if headers is None:
        headers = HEADERS_JSON
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → GET {url}")
            r = requests.get(url, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def http_post(url, body=None, headers=None):
    if headers is None:
        headers = HEADERS_POST
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → POST {url}")
            r = requests.post(url, json=body, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def save_artifact(step, filename, data, binary=False):
    """Save a test artifact to the run directory, prefixed with the step number."""
    path = os.path.join(RUN_DIR, f"step{step:03d}_{filename}")
    if binary:
        with open(path, "wb") as f:
            f.write(data)
    else:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  ✓ {path}")
    return path

def print_keys(data, *keys):
    subset = {k: data.get(k) for k in keys if k in data}
    print(json.dumps(subset, indent=2, ensure_ascii=False))

print(f"Configuration OK — sortie dans : {RUN_DIR}")


Configuration OK — sortie dans : runs/05_iti98_lookup_20260311T211153


---
## Étapes 0–30 — Lookup du code DA63 en CIM-11 (GET)

**Requête** : `GET /fhir/CodeSystem/$lookup?system=https://smt.esante.gouv.fr/terminologie-cim11-mms&code=DA63&_format=json`
**Objectif** : Vérifier que `DA63` a pour libellé "Ulcère du duodénum".

In [2]:
# Étape 0  — Construction de la requête
# Étape 10 — TRANSACTION : GET $lookup DA63
CIM11_SYSTEM = "https://smt.esante.gouv.fr/terminologie-cim11-mms"

url_lookup = f"{FHIR_BASE}/CodeSystem/$lookup?system={quote(CIM11_SYSTEM)}&code=DA63&_format=json"
r_lookup = http_get(url_lookup)
result = r_lookup.json()
save_artifact(30, "lookup-da63.json", result)

# Étape 20 — Réception et intégration
params = {p["name"]: p for p in result.get("parameter", [])}
display_val = params.get("display", {}).get("valueString", "")

# Étape 30 — PREUVE
print("[PREUVE étape 30] $lookup DA63 :")
print(f"  code    : DA63")
print(f"  display : {display_val}")
assert "duod" in display_val.lower(), f"Libellé inattendu : {display_val!r}"
print("  ✓ Libellé contient 'duod' (Ulcère du duodénum)")


  → GET https://gazelle-proxypatans.kereval.cloud:10102/fhir/CodeSystem/$lookup?system=https%3A//smt.esante.gouv.fr/terminologie-cim11-mms&code=DA63&_format=json
  ✓ runs/05_iti98_lookup_20260311T211153/step030_lookup-da63.json
[PREUVE étape 30] $lookup DA63 :
  code    : DA63
  display : Ulcère du duodénum
  ✓ Libellé contient 'duod' (Ulcère du duodénum)


---
## Étapes 40–70 — Lookup avec codes enfants (GET + propriété `child`)

**Requête** : `GET /fhir/CodeSystem/$lookup?system=...cim11...&code=DA63&property=child&_format=json`

In [3]:
# Étape 40 — Construction de la requête
# Étape 50 — TRANSACTION : GET $lookup DA63 avec property=child
url_lookup_ch = f"{FHIR_BASE}/CodeSystem/$lookup?system={quote(CIM11_SYSTEM)}&code=DA63&property=child&_format=json"
r_ch = http_get(url_lookup_ch)
result_ch = r_ch.json()
save_artifact(70, "lookup-da63-children.json", result_ch)

# Étape 60 — Réception et intégration
children = [
    p for p in result_ch.get("parameter", [])
    if p.get("name") == "property"
    and any(pp.get("name") == "code" and pp.get("valueCode") == "child"
            for pp in p.get("part", []))
]

# Étape 70 — PREUVE
print(f"[PREUVE étape 70] Codes enfants de DA63 : {len(children)} trouvé(s)")
for child in children:
    parts = {pp["name"]: pp for pp in child.get("part", [])}
    print(f"  {parts.get('value', {}).get('valueCode', '')} — {parts.get('description', {}).get('valueString', '')}")


  → GET https://gazelle-proxypatans.kereval.cloud:10102/fhir/CodeSystem/$lookup?system=https%3A//smt.esante.gouv.fr/terminologie-cim11-mms&code=DA63&property=child&_format=json
  ✓ runs/05_iti98_lookup_20260311T211153/step070_lookup-da63-children.json
[PREUVE étape 70] Codes enfants de DA63 : 9 trouvé(s)
  DA63.Z — 
  DA63.2 — 
  DA63.1 — 
  DA63.Y — 
  DA63.0 — 
  DA63.3 — 
  DA63.4 — 
  DA63.6 — 
  DA63.5 — 


---
## Étapes 80–110 — Lookup du code SNOMED 425803006 (POST)

**Requête** : `POST /fhir/CodeSystem/$lookup`
**Objectif** : Vérifier que `425803006` a pour display "région thoracique" et afficher si ce code est inactif.

In [4]:
# Étape 80 — Construction de la requête
# Étape 90 — TRANSACTION : POST $lookup 425803006 en SNOMED CT
SNOMED_SYSTEM = "http://snomed.info/sct"
HEADERS_POST_JSON = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
lookup_body = {
    "resourceType": "Parameters",
    "parameter": [
        {"name": "system",   "valueUri":  SNOMED_SYSTEM},
        {"name": "code",     "valueCode": "425803006"},
        {"name": "property", "valueCode": "inactive"},
    ],
}

r_snomed = http_post(f"{FHIR_BASE}/CodeSystem/$lookup", body=lookup_body, headers=HEADERS_POST_JSON)

# Étape 100 — Réception et intégration
try:
    result_s = r_snomed.json()
    save_artifact(110, "lookup-snomed-425803006.json", result_s)
except Exception:
    print(f"⚠ Réponse non-JSON reçue")
    save_artifact(110, "lookup-snomed-425803006.html", r_snomed.content, binary=True)
    result_s = None

# Étape 110 — PREUVE
if result_s:
    params_s   = {p["name"]: p for p in result_s.get("parameter", [])}
    display_s  = params_s.get("display", {}).get("valueString", "")
    inactive_s = None
    for p in result_s.get("parameter", []):
        if p.get("name") == "property":
            parts = {pp["name"]: pp for pp in p.get("part", [])}
            if parts.get("code", {}).get("valueCode") == "inactive":
                inactive_s = parts.get("value", {}).get("valueBoolean")
    print("[PREUVE étape 110] $lookup SNOMED 425803006 (POST) :")
    print(f"  display  : {display_s}")
    print(f"  inactive : {inactive_s}")
else:
    print("[PREUVE étape 110] POST $lookup SNOMED 425803006 envoyé — réponse non-JSON")

  → POST https://gazelle-proxypatans.kereval.cloud:10102/fhir/CodeSystem/$lookup
⚠ Réponse non-JSON reçue
  ✓ runs/05_iti98_lookup_20260311T211153/step110_lookup-snomed-425803006.html
[PREUVE étape 110] POST $lookup SNOMED 425803006 envoyé — réponse non-JSON


---
## Récapitulatif

In [5]:
print(f"Run : {RUN_DIR}")
print(f"{'Fichier':<45} {'Taille':>10}")
print("-" * 57)
for fname in sorted(os.listdir(RUN_DIR)):
    fpath = os.path.join(RUN_DIR, fname)
    size  = os.path.getsize(fpath)
    size_str = f"{size/1024:.1f} KB" if size < 1_000_000 else f"{size/1024/1024:.1f} MB"
    print(f"  {fname:<43} {size_str:>10}")
print(f"\n✓ {NOTEBOOK_ID} — terminé.")


Run : runs/05_iti98_lookup_20260311T211153
Fichier                                           Taille
---------------------------------------------------------
  step030_lookup-da63.json                        3.6 KB
  step070_lookup-da63-children.json               2.2 KB
  step110_lookup-snomed-425803006.html          128.7 KB

✓ 05_iti98_lookup — terminé.
